# Feature Engineering — Fraud_Data.csv
**Objective:** Create behavioural, temporal, and geolocation features that capture fraud signals invisible in raw fields.


In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings = lambda *a, **k: None
import warnings; warnings.filterwarnings('ignore')

from src.data_loader import load_fraud_data, load_ip_country
from src.preprocessor import merge_ip_country
from src.feature_engineering import engineer_features

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120


## 1. Load and Merge

In [ ]:
fraud_df = load_fraud_data('../data/raw/Fraud_Data.csv')
ip_country_df = load_ip_country('../data/raw/IpAddress_to_Country.csv')

# Merge geolocation
fraud_geo = merge_ip_country(fraud_df, ip_country_df)
print("After IP merge:", fraud_geo.shape)


## 2. Apply Feature Engineering

In [ ]:
fraud_engineered = engineer_features(fraud_geo)
print("New columns created:")
new_cols = ['time_since_signup_hours', 'hour_of_day', 'day_of_week',
            'transaction_count_1h', 'transaction_count_24h']
print(fraud_engineered[new_cols].describe())


## 3. time_since_signup_hours Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Overall distribution
axes[0].hist(fraud_engineered['time_since_signup_hours'].clip(0, 2000),
             bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Time Since Signup Distribution')
axes[0].set_xlabel('Hours')

# By class
for label, name, color in [(0, 'Legitimate', 'steelblue'), (1, 'Fraud', 'tomato')]:
    sub = fraud_engineered[fraud_geo['class'] == label]['time_since_signup_hours'].clip(0, 500)
    axes[1].hist(sub, bins=60, alpha=0.6, label=name, color=color, density=True)
axes[1].set_title('Time Since Signup by Class (density)')
axes[1].set_xlabel('Hours (clipped at 500)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/time_since_signup.png', bbox_inches='tight')
plt.show()

# Key stats
print("Median time_since_signup — Fraud:", 
      fraud_engineered[fraud_geo['class']==1]['time_since_signup_hours'].median().round(1), "hrs")
print("Median time_since_signup — Legit:", 
      fraud_engineered[fraud_geo['class']==0]['time_since_signup_hours'].median().round(1), "hrs")


## 4. Hourly Fraud Pattern

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
fraud_rate_by_hour = fraud_geo.copy()
fraud_rate_by_hour['hour_of_day'] = pd.to_datetime(fraud_geo['purchase_time']).dt.hour
rate = fraud_rate_by_hour.groupby('hour_of_day')['class'].mean() * 100

ax.bar(rate.index, rate.values, color='slateblue', edgecolor='white')
ax.set_title('Fraud Rate by Hour of Day', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of Day (0=midnight)')
ax.set_ylabel('Fraud Rate (%)')
ax.set_xticks(range(24))
plt.tight_layout()
plt.savefig('../reports/figures/fraud_by_hour.png', bbox_inches='tight')
plt.show()


## 5. Transaction Velocity Features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, label in [(axes[0], 'transaction_count_1h', '1-Hour Window'),
                        (axes[1], 'transaction_count_24h', '24-Hour Window')]:
    for cls, name, color in [(0, 'Legit', 'steelblue'), (1, 'Fraud', 'tomato')]:
        sub = fraud_engineered[fraud_geo['class'] == cls][col]
        ax.hist(sub, bins=range(0, int(sub.max())+2), alpha=0.6, label=name, color=color, density=True)
    ax.set_title(f'Transaction Count ({label})')
    ax.set_xlabel('Prior Transaction Count')
    ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/velocity_features.png', bbox_inches='tight')
plt.show()


## 6. Feature Engineering Summary

| Feature | Construction | Fraud Signal Rationale |
|---|---|---|
| `time_since_signup_hours` | `(purchase_time - signup_time) / 3600` | Fraudsters often buy immediately after creating throwaway accounts |
| `hour_of_day` | `purchase_time.dt.hour` | Fraud concentrates in off-hours (0–6am) |
| `day_of_week` | `purchase_time.dt.dayofweek` | Weekend patterns differ from weekday |
| `transaction_count_1h` | Sorted rolling count per user | Burst buying within 1 hour is a strong fraud signal |
| `transaction_count_24h` | Same, 24h window | Daily velocity — catches high-frequency fraud accounts |
| `country` (from IP) | Range-based IP lookup | Certain geographies show systematically higher fraud rates |
